# GWS 2022 Analysis

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px 

### Connecting to database

In [2]:
db_path = "data/nba.sqlite"  # change to your path
conn = sqlite3.connect(db_path)

tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()
print([t[0] for t in tables])

['game', 'game_summary', 'other_stats', 'officials', 'inactive_players', 'game_info', 'line_score', 'play_by_play', 'player', 'team', 'common_player_info', 'team_details', 'team_history', 'draft_combine_stats', 'draft_history', 'team_info_common']


### Retreiving data
Getting both home and away game for the 2022 season

In [3]:
query = """
SELECT *
FROM game as g
JOIN other_stats AS o ON g.game_id = o.game_id
JOIN game_info as i ON g.game_id = i.game_id
WHERE g.team_abbreviation_home LIKE "%GSW%"
AND g.season_id = "22021"
"""
df_home = pd.read_sql_query(query, conn)
df_home.head()

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,pts_fb_away,largest_lead_away,team_turnovers_away,total_turnovers_away,team_rebounds_away,pts_off_to_away,game_id,game_date,attendance,game_time
0,22021,1610612744,GSW,Golden State Warriors,0022100016,2021-10-21 00:00:00,GSW vs. LAC,W,240,43.0,...,25,18,0,21,9,18,0022100016,2021-10-21 00:00:00,18064,2:28
1,22021,1610612744,GSW,Golden State Warriors,0022100086,2021-10-30 00:00:00,GSW vs. OKC,W,240,39.0,...,16,0,1,16,10,12,0022100086,2021-10-30 00:00:00,18064,2:02
2,22021,1610612744,GSW,Golden State Warriors,0022100117,2021-11-03 00:00:00,GSW vs. CHA,W,240,42.0,...,19,22,1,17,6,17,0022100117,2021-11-03 00:00:00,18064,2:15
3,22021,1610612744,GSW,Golden State Warriors,0022100130,2021-11-05 00:00:00,GSW vs. NOP,W,240,47.0,...,11,7,2,19,10,26,0022100130,2021-11-05 00:00:00,18064,2:13
4,22021,1610612744,GSW,Golden State Warriors,0022100145,2021-11-07 00:00:00,GSW vs. HOU,W,240,44.0,...,36,21,2,17,7,28,0022100145,2021-11-07 00:00:00,18064,2:12


In [4]:
query = """
SELECT *
FROM game AS g
JOIN other_stats AS o ON g.game_id = o.game_id
JOIN game_info as i ON g.game_id = i.game_id
WHERE matchup_home LIKE "%GSW%"
    AND g.season_id = "22021"
    AND g.team_abbreviation_home NOT LIKE "%GSW%"

"""
df_away = pd.read_sql_query(query, conn)
df_away.head()

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,pts_fb_away,largest_lead_away,team_turnovers_away,total_turnovers_away,team_rebounds_away,pts_off_to_away,game_id,game_date,attendance,game_time
0,22021,1610612747,LAL,Los Angeles Lakers,0022100002,2021-10-19 00:00:00,LAL vs. GSW,L,240,45.0,...,20,10,1,18,13,20,0022100002,2021-10-19 00:00:00,18997,2:33
1,22021,1610612758,SAC,Sacramento Kings,0022100039,2021-10-24 00:00:00,SAC vs. GSW,L,240,42.0,...,16,12,1,7,10,9,0022100039,2021-10-24 00:00:00,13876,2:17
2,22021,1610612760,OKC,Oklahoma City Thunder,0022100051,2021-10-26 00:00:00,OKC vs. GSW,L,240,36.0,...,4,10,0,15,7,18,0022100051,2021-10-26 00:00:00,15717,2:11
3,22021,1610612766,CHA,Charlotte Hornets,0022100194,2021-11-14 00:00:00,CHA vs. GSW,W,240,42.0,...,10,7,2,13,11,17,0022100194,2021-11-14 00:00:00,19559,2:15
4,22021,1610612751,BKN,Brooklyn Nets,0022100210,2021-11-16 00:00:00,BKN vs. GSW,L,240,34.0,...,25,28,1,22,8,20,0022100210,2021-11-16 00:00:00,17732,2:23


In [5]:
query = """
SELECT *
FROM game AS g
JOIN other_stats AS o ON g.game_id = o.game_id
JOIN game_info as i ON g.game_id = i.game_id
WHERE matchup_home  NOT LIKE "%GSW%"
    AND g.season_id = "22021"

"""
df_others = pd.read_sql_query(query, conn)
df_others.head()

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,pts_fb_away,largest_lead_away,team_turnovers_away,total_turnovers_away,team_rebounds_away,pts_off_to_away,game_id,game_date,attendance,game_time
0,22021,1610612749,MIL,Milwaukee Bucks,0022100001,2021-10-19 00:00:00,MIL vs. BKN,W,240,48.0,...,21,23,1,8,8,2,0022100001,2021-10-19 00:00:00,17341.0,2:20
1,22021,1610612757,POR,Portland Trail Blazers,0022100013,2021-10-20 00:00:00,POR vs. SAC,L,240,45.0,...,2,3,1,13,8,18,0022100013,2021-10-20 00:00:00,17467.0,2:18
2,22021,1610612756,PHX,Phoenix Suns,0022100012,2021-10-20 00:00:00,PHX vs. DEN,L,240,36.0,...,10,16,0,18,5,27,0022100012,2021-10-20 00:00:00,16074.0,2:09
3,22021,1610612750,MIN,Minnesota Timberwolves,0022100008,2021-10-20 00:00:00,MIN vs. HOU,W,240,42.0,...,13,4,0,24,13,38,0022100008,2021-10-20 00:00:00,16079.0,2:13
4,22021,1610612763,MEM,Memphis Grizzlies,0022100007,2021-10-20 00:00:00,MEM vs. CLE,W,240,53.0,...,22,16,2,12,4,12,0022100007,2021-10-20 00:00:00,15975.0,2:07


### Removing duplicated columns and merging dfs

In [6]:
df_home = df_home.loc[:, ~df_home.columns.duplicated()]
df_away = df_away.loc[:, ~df_away.columns.duplicated()]
df_others = df_others.loc[:, ~df_others.columns.duplicated()]

df_home["home"] = 1
df_away["home"] = 0 


df = pd.concat([df_home, df_away], ignore_index=True)
df.head()

,season_id,team_id_home,team_abbreviation_home,team_name_home,game_id,game_date,matchup_home,wl_home,min,fgm_home,...,pts_2nd_chance_away,pts_fb_away,largest_lead_away,team_turnovers_away,total_turnovers_away,team_rebounds_away,pts_off_to_away,attendance,game_time,home
0,22021,1610612744,GSW,Golden State Warriors,0022100016,2021-10-21 00:00:00,GSW vs. LAC,W,240,43.0,...,7,25,18,0,21,9,18,18064,2:28,1
1,22021,1610612744,GSW,Golden State Warriors,0022100086,2021-10-30 00:00:00,GSW vs. OKC,W,240,39.0,...,5,16,0,1,16,10,12,18064,2:02,1
2,22021,1610612744,GSW,Golden State Warriors,0022100117,2021-11-03 00:00:00,GSW vs. CHA,W,240,42.0,...,9,19,22,1,17,6,17,18064,2:15,1
3,22021,1610612744,GSW,Golden State Warriors,0022100130,2021-11-05 00:00:00,GSW vs. NOP,W,240,47.0,...,11,11,7,2,19,10,26,18064,2:13,1
4,22021,1610612744,GSW,Golden State Warriors,0022100145,2021-11-07 00:00:00,GSW vs. HOU,W,240,44.0,...,6,36,21,2,17,7,28,18064,2:12,1


In [7]:
df.columns

Index(['season_id', 'team_id_home', 'team_abbreviation_home', 'team_name_home',
       'game_id', 'game_date', 'matchup_home', 'wl_home', 'min', 'fgm_home',
       'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home', 'fg3_pct_home',
       'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home', 'dreb_home',
       'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home', 'pf_home',
       'pts_home', 'plus_minus_home', 'video_available_home', 'team_id_away',
       'team_abbreviation_away', 'team_name_away', 'matchup_away', 'wl_away',
       'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away',
       'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away',
       'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away',
       'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away',
       'season_type', 'league_id', 'team_city_home', 'pts_paint_home',
       'pts_2nd_chance_home', 'pts_fb_home', 'largest_lead_home',
       'lead_changes', '

### Exploring the performance of GWS

In [8]:
df["win"] = np.where(
    ((df["team_abbreviation_home"] == "GSW") & (df["wl_home"] == "W")) |
    ((df["team_abbreviation_home"] != "GSW") & (df["wl_home"] == "L")),
    1,
    0,
 )

df_others["win"] = np.where(
    (df_others["wl_home"] == "W"), 1, 0 
)



In [9]:
df.groupby(by = "home").agg(
    win_total = ("win","sum"),
    total_games = ("home", "count")
)

,win_total,total_games
home,,
0,17,33
1,29,35


In [10]:
df["win_flag"] = df["win"].map({1: "1", 0: "0"})

fig = px.scatter(
    df,
    x="fg_pct_home",
    y="fg_pct_away",
    color="win_flag",
    color_discrete_map={
        "1": "#1f77b4",  # home/away win color
        "0": "#ff7f0e",  # loss color
    },
    category_orders={"win_flag": ["1", "0"]},
    labels={"win_flag": "win"},
)
fig.update_layout(xaxis_title="Home FG%", yaxis_title="Away FG%")
fig.show()

## Determinants of shot efficiency

In [11]:
df.select_dtypes(include="number").columns

Index(['min', 'fgm_home', 'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home',
       'fg3_pct_home', 'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home',
       'dreb_home', 'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home',
       'pf_home', 'pts_home', 'plus_minus_home', 'video_available_home',
       'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away',
       'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away',
       'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away',
       'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away',
       'pts_paint_home', 'pts_2nd_chance_home', 'pts_fb_home',
       'largest_lead_home', 'lead_changes', 'times_tied',
       'team_turnovers_home', 'total_turnovers_home', 'team_rebounds_home',
       'pts_off_to_home', 'pts_paint_away', 'pts_2nd_chance_away',
       'pts_fb_away', 'largest_lead_away', 'team_turnovers_away',
       'total_turnovers_away', 'team_rebounds_away', 'pts_off_t

In [12]:
gsw_home = df[df["team_abbreviation_home"] == "GSW"].copy()
gsw_home = gsw_home.select_dtypes(include="number")
gsw_home.columns

Index(['min', 'fgm_home', 'fga_home', 'fg_pct_home', 'fg3m_home', 'fg3a_home',
       'fg3_pct_home', 'ftm_home', 'fta_home', 'ft_pct_home', 'oreb_home',
       'dreb_home', 'reb_home', 'ast_home', 'stl_home', 'blk_home', 'tov_home',
       'pf_home', 'pts_home', 'plus_minus_home', 'video_available_home',
       'fgm_away', 'fga_away', 'fg_pct_away', 'fg3m_away', 'fg3a_away',
       'fg3_pct_away', 'ftm_away', 'fta_away', 'ft_pct_away', 'oreb_away',
       'dreb_away', 'reb_away', 'ast_away', 'stl_away', 'blk_away', 'tov_away',
       'pf_away', 'pts_away', 'plus_minus_away', 'video_available_away',
       'pts_paint_home', 'pts_2nd_chance_home', 'pts_fb_home',
       'largest_lead_home', 'lead_changes', 'times_tied',
       'team_turnovers_home', 'total_turnovers_home', 'team_rebounds_home',
       'pts_off_to_home', 'pts_paint_away', 'pts_2nd_chance_away',
       'pts_fb_away', 'largest_lead_away', 'team_turnovers_away',
       'total_turnovers_away', 'team_rebounds_away', 'pts_off_t

In [13]:
gsw_home = gsw_home.drop(
    columns=["min", "home", "win", "video_available_home", "video_available_away", "pts_home", 
    "fgm_home", "fg3_pct_home", "fg3m_home"],
    errors="ignore",
)
corr = gsw_home.corr()


In [14]:
fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Correlation Matrix"
)

fig.show()

In [15]:

target = "fg_pct_home"


corr_with_target = (
    gsw_home.corr(numeric_only=True)[target]
    .drop(target)
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

top_n = 15
corr_top = corr_with_target.head(top_n).reset_index()
corr_top.columns = ["feature", "corr"]

fig = px.bar(
    corr_top,
    x="corr",
    y="feature",
    orientation="h",
    title=f"Top {top_n} correlations with {target}",
)
fig.update_layout(template="plotly_dark")
fig.show()